# Task 3 part 1: Data preparation

Goal: select a diverse, reliable subset of 50 perturbations and compute their mean log2 fold-change profile ("fingerprint") per condition, relative to control cells in the same condition. 10 of these 50 are held out purely for testing generalization to perturbations never seen during training/tuning.

In [1]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd

In [2]:
DATA_DIR = "/home/ubuntu/data/frangieh"
adata = sc.read_h5ad(f"{DATA_DIR}/rna_qc_filtered.h5ad")
adata

AnnData object with n_obs × n_vars = 213870 × 23711
    obs: 'library_preparation_protocol', 'perturbation_2', 'MOI', 'sgRNA', 'UMI_count', 'guide_id', 'perturbation', 'tissue_type', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation_type_2', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet'
    var: 'ensembl_id', 'ncounts', 'ncells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'

## Normalize

Keep raw counts and the normalized-but-not-logged counts around as separate layers. The log2 fold-change fingerprint below is computed on the normalized (non-log) scale, matching how fold changes are conventionally defined; the log1p version stays in `.X` for everything else (HVG selection, later modeling).

In [3]:
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
adata.layers["norm"] = adata.X.copy()

sc.pp.log1p(adata)

## Restrict to highly variable genes

Working with all ~23,700 genes would make each fingerprint mostly noise and unnecessarily expensive to compute and model. Restrict to the top 2000 HVGs (same choice as Task 2) so fingerprints are a manageable size and comparable across notebooks.

In [4]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
adata = adata[:, adata.var["highly_variable"]].copy()

# only 2000 genes now, so densify the normalized layer for fast row-masking below
adata.layers["norm"] = np.asarray(adata.layers["norm"].todense())

adata.shape

(213870, 2000)

## Filter out perturbations with too few cells

A fingerprint averaged over very few cells is noisy. Require at least 100 cells in *every* condition for a perturbation to be considered reliable enough to model.

In [5]:
MIN_CELLS_PER_CONDITION = 100

cell_counts = (
    adata.obs.groupby(["perturbation", "perturbation_2"], observed=True)
    .size()
    .unstack(fill_value=0)
)
cell_counts = cell_counts.drop(index="control")

min_cells = cell_counts.min(axis=1)
reliable_perturbations = min_cells[min_cells >= MIN_CELLS_PER_CONDITION].index.tolist()

print(f"{len(reliable_perturbations)} of {len(cell_counts)} perturbations retained")

196 of 248 perturbations retained


## Compute pseudobulk log2 fold-change fingerprints

For each (perturbation, condition) pair, compare the mean normalized expression of knockout cells to the mean of control cells in the *same* condition. A pseudocount of 1 (matching the log1p convention used elsewhere) avoids division by zero for genes with near-zero expression.

In [6]:
PSEUDOCOUNT = 1.0
conditions = adata.obs["perturbation_2"].unique().tolist()
norm = adata.layers["norm"]

# mean normalized expression of control cells, per condition
control_means = {}
for cond in conditions:
    mask = (adata.obs["perturbation_2"] == cond) & (adata.obs["perturbation"] == "control")
    control_means[cond] = norm[mask.values].mean(axis=0)

# log2FC fingerprint per (perturbation, condition), relative to control cells of that condition
fingerprints = {}
for pert in reliable_perturbations:
    for cond in conditions:
        mask = (adata.obs["perturbation_2"] == cond) & (adata.obs["perturbation"] == pert)
        if mask.sum() == 0:
            continue
        pert_mean = norm[mask.values].mean(axis=0)
        fingerprints[(pert, cond)] = np.log2((pert_mean + PSEUDOCOUNT) / (control_means[cond] + PSEUDOCOUNT))

fingerprint_index = pd.MultiIndex.from_tuples(fingerprints.keys(), names=["perturbation", "condition"])
fingerprints_df = pd.DataFrame(np.vstack(list(fingerprints.values())), index=fingerprint_index, columns=adata.var_names)

fingerprints_df.shape

(588, 2000)

## Bring in Task 2 cluster labels for diverse selection

Use the Co-culture Leiden clusters from Task 2 (`perturbation_groups.csv`) as a reference for how similar/different perturbations' effects are. Co-culture is used because it's the most granular clustering (11 clusters) and the one condition every group size is required to include. This lets us pick perturbations that span the real diversity of effects rather than clustering around one dominant phenotype.

In [7]:
groups = pd.read_csv(f"{DATA_DIR}/perturbation_groups.csv")
coculture_leiden = groups[(groups["condition"] == "Co-culture") & (groups["method"] == "leiden")]
coculture_leiden = coculture_leiden.set_index("perturbation")["cluster"]

cluster_labels = coculture_leiden.reindex(reliable_perturbations).dropna()
missing = set(reliable_perturbations) - set(cluster_labels.index)
print(f"{len(missing)} reliable perturbations missing a cluster label: {missing}")

cluster_labels.value_counts().sort_index()

0 reliable perturbations missing a cluster label: set()


cluster
0      2
1     12
2     17
3      8
4      2
5     21
6     24
7     38
8      7
9     60
10     5
Name: count, dtype: int64

## Stratified selection: 50 perturbations, then a 10-perturbation held-out test set

Sample proportionally from each cluster so the 50 chosen genes mirror the overall cluster distribution, then repeat the same proportional sampling *within* the 50 to pick the 10 held out purely for testing. This keeps both the modeling set and the untouchable test set representative of the full diversity of effects, rather than accidentally easy or accidentally hard.

In [8]:
rng = np.random.default_rng(42)


def stratified_sample(labels, n, rng):
    """Proportionally sample n items across the groups in `labels` (a Series of cluster labels)."""
    frac = n / len(labels)
    chosen = []
    for _, group in labels.groupby(labels):
        k = min(len(group), max(1, round(len(group) * frac)))
        chosen.extend(rng.choice(group.index, size=k, replace=False))
    chosen = rng.choice(chosen, size=min(n, len(chosen)), replace=False)
    return list(chosen)


selected_50 = stratified_sample(cluster_labels, 50, rng)
held_out_10 = stratified_sample(cluster_labels.loc[selected_50], 10, rng)
train_40 = [p for p in selected_50 if p not in held_out_10]

print(f"selected: {len(selected_50)}, train: {len(train_40)}, held out: {len(held_out_10)}")
print("held out perturbations:", sorted(held_out_10))

selected: 50, train: 40, held out: 10
held out perturbations: [np.str_('CD151'), np.str_('CTSB'), np.str_('HLA-F'), np.str_('LCP1'), np.str_('LRPAP1'), np.str_('NT5E'), np.str_('S100B'), np.str_('SCARB2'), np.str_('SERPINA3'), np.str_('TYR')]


## Save outputs for the modeling notebook

In [9]:
fingerprints_selected = fingerprints_df.loc[pd.IndexSlice[selected_50, :], :]

fingerprints_selected.to_pickle(f"{DATA_DIR}/task3_fingerprints_50.pkl")
pd.Series(train_40, name="perturbation").to_csv(f"{DATA_DIR}/task3_train_40.csv", index=False)
pd.Series(held_out_10, name="perturbation").to_csv(f"{DATA_DIR}/task3_held_out_10.csv", index=False)

print("saved fingerprints and train/held-out perturbation lists")

saved fingerprints and train/held-out perturbation lists
